# VAKOG Wine Review Authorship — Team Rhone (revised)

**Research question:** Can VAKOG sensory language identify who wrote a wine review, and what does each critic's sensory "fingerprint" look like?

**What changed from v1**

| Fix | Why it matters |
|---|---|
| Deduplicate reviews, drop Twitter handle | Identical reviews in train *and* test inflate scores; the handle *is* the label |
| Mask each wine's own place / variety / winery names | Stops "Barolo" or "Red Blend" from leaking authorship or being counted as Visual |
| Fixed VAKOG lexicon (no term in two channels, colour+food phrases handled) | `orange`, `violet`, `red fruit` were double-counted or mis-channelled |
| Rates per word + log-ratio (clr) + VAKOG-tagged terms | 5 ratios that sum to 1 carry too little information to separate ~15 authors |
| Split grouped by winery | Prevents memorising wineries the same critic reviews every year |
| Elastic net (profiles) + XGBoost (accuracy ceiling) replace the neural net | Interpretable coefficients = reviewer profiles |
| Macro F1 (optimizing) + per-author recall floor (satisficing) | Simple, imbalance-aware |
| Equalized-odds audit across score bands and price tiers | Winetitude's no-bias stance |
| Within-California check | Separates *style* from *which region the critic covers* |

## 1. Imports and reproducibility

In [ ]:
!pip install -q kagglehub xgboost

import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp

import kagglehub

from sklearn import __version__ as SKLEARN_VERSION
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    ConfusionMatrixDisplay,
)

from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Download and load the dataset

In [ ]:
import kagglehub
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("zynicide/wine-reviews"))

INPUT_FILE = path / "winemag-data-130k-v2.csv"

print("Dataset directory:", path)

print("Path to dataset files:", INPUT_FILE)

In [ ]:
df = pd.read_csv(INPUT_FILE)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Cleaning and deduplication

* Rows without text or author can't be used.
* **Exact duplicates** (same text, same author) are removed so a review can't appear in both train and test.
* **Ambiguous duplicates** (same text credited to *different* authors) are dropped entirely: the label is unreliable.
* `taster_twitter_handle` is removed: it is the author's identity in another form (target leakage).

In [ ]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
print("Raw rows:", len(df))

df = df.dropna(subset=["description", "taster_name"]).copy()
df["description"] = df["description"].astype(str).str.strip()
df["taster_name"] = df["taster_name"].astype(str)
print("Rows with text + author:", len(df))

# Same review, same author, more than once -> keep one copy
df = df.drop_duplicates(subset=["description", "taster_name"])
print("After removing exact duplicates:", len(df))

# Same review credited to different authors -> label is ambiguous, drop every copy
ambiguous = df["description"].duplicated(keep=False)
df = df[~ambiguous]
print("After removing ambiguous-author duplicates:", len(df))

# The Twitter handle IS the author -> remove so it can never become a feature
df = df.drop(columns=["taster_twitter_handle"], errors="ignore")
df = df.reset_index(drop=True)

display(df["taster_name"].value_counts())

## 3b. Mask each wine's own names

Every review is checked against **its own** metadata (country, province, region, variety, winery, designation). Those names are replaced with `NAME`.

This stops two problems:
1. Geography leaking authorship ("Barolo" → Kerin O'Keefe).
2. Names being mis-read as sensory words ("Red Blend", "Black Label" → Visual).

In [ ]:
MASK_COLUMNS = ["country", "province", "region_1", "region_2", "variety", "winery", "designation"]


def mask_metadata(row):
    """Replace this wine's own place / variety / winery names with the token NAME."""
    text = row["description"]

    names = {
        str(row[col])
        for col in MASK_COLUMNS
        if pd.notna(row[col]) and len(str(row[col])) >= 3
    }

    # Longest names first so "Russian River Valley" is masked before "Valley"
    for name in sorted(names, key=len, reverse=True):
        text = re.sub(
            r"(?<!\w)" + re.escape(name) + r"(?!\w)",
            " NAME ",
            text,
            flags=re.IGNORECASE,
        )
    return text


df["text_masked"] = df.apply(mask_metadata, axis=1)

display(df[["description", "text_masked"]].head(3))

## 4. VAKOG lexicon (fixed)

* **V** Visual · **A** Auditory · **K** Kinesthetic / mouthfeel · **O** Olfactory · **G** Gustatory
* **Cue words** (`aromas`, `palate`, `finish`) are no longer *counted* as sensory terms. They decide whether a shared descriptor like `cherry` is O or G, and they are kept as `CUE:` style tokens.
* `orange` and `violet` were in both Visual and Shared → now only Shared.
* `bright` and `deep` were removed from Visual: in reviews they almost always describe acidity or flavour intensity.
* Colour + food phrases (`red fruit`, `black cherry`, `white pepper`, `green herbs`) are handled as **one** shared descriptor, so `red`/`black` aren't also counted as Visual.

In [ ]:
VISUAL_TERMS = [
    "pale", "ruby", "garnet", "golden", "gold", "straw", "amber", "copper", "salmon",
    "opaque", "translucent", "clear", "cloudy", "inky", "purple",
    "red", "white", "green", "yellow", "pink", "brown", "black", "dark",
    "color", "colour", "colored", "hue", "hued", "rim", "appearance",
]

AUDITORY_TERMS = [
    "loud", "quiet", "whisper", "whispers", "sound", "sounds", "noisy", "ringing",
]

KINESTHETIC_TERMS = [
    "smooth", "rough", "silky", "velvety", "creamy", "chewy", "firm", "soft",
    "hard", "heavy", "light", "crisp", "brisk", "tannic", "tannin", "tannins",
    "texture", "textured", "mouthfeel", "grip", "grippy", "dense", "lean",
    "weighty", "supple", "round", "full-bodied", "medium-bodied", "light-bodied",
]

# Cue words: decide O vs G context; kept as CUE tokens, not counted
OLFACTORY_CUES = ["aroma", "aromas", "nose", "bouquet", "scent", "scents", "smell", "smells", "whiff"]
GUSTATORY_CUES = ["palate", "flavor", "flavors", "flavour", "flavours", "taste", "tastes",
                  "finish", "aftertaste", "mouth"]

# Descriptors that are always olfactory / always gustatory
OLFACTORY_TERMS = ["aromatic", "fragrant", "fragrance", "perfume", "perfumed", "scented", "musky"]
DIRECT_GUSTATORY_TERMS = [
    "sweet", "sweetness", "bitter", "bitterness", "sour", "salty", "acid",
    "acidic", "acidity", "tart", "savory", "dry", "off-dry",
]

# Can be smell OR taste: context decides
SHARED_AROMA_FLAVOR_TERMS = [
    "tropical fruit", "dried herb", "fresh herb", "stone fruit", "citrus fruit", "candied fruit",
    "apple", "pear", "peach", "apricot", "plum", "cherry", "blackberry",
    "raspberry", "strawberry", "blueberry", "cranberry", "pineapple", "mango",
    "melon", "lemon", "lime", "orange", "grapefruit", "citrus",
    "sage", "thyme", "mint", "basil", "herb", "herbs",
    "pepper", "vanilla", "cinnamon", "clove", "spice", "spices",
    "chocolate", "coffee", "caramel", "honey",
    "earth", "earthy", "leather", "tobacco", "smoke", "smoky",
    "flower", "flowers", "floral", "rose", "violet",
    "broom", "brimstone",
]

In [ ]:
# Guard rail: no term may sit in two lists (this is how "orange" and "violet" were double-counted)
LEXICON = {
    "V": VISUAL_TERMS, "A": AUDITORY_TERMS, "K": KINESTHETIC_TERMS,
    "O": OLFACTORY_TERMS, "G": DIRECT_GUSTATORY_TERMS, "SHARED": SHARED_AROMA_FLAVOR_TERMS,
    "O_CUE": OLFACTORY_CUES, "G_CUE": GUSTATORY_CUES,
}

owner = {}
conflicts = []
for list_name, terms in LEXICON.items():
    for term in terms:
        if term in owner and owner[term] != list_name:
            conflicts.append((term, owner[term], list_name))
        owner[term] = list_name

assert not conflicts, f"Terms in more than one list: {conflicts}"
print(f"Lexicon OK: {len(owner)} unique terms, none in two lists.")

## 5. VAKOG parser

For each clause the parser:
1. finds cue words (context),
2. removes colour words that **name** a wine ("red wine", "this white"),
3. pulls out shared descriptors (colour+food phrases first) and assigns them O, G, or OG (ambiguous, split 0.5/0.5),
4. counts single-channel descriptors in what is left.

It returns **counts** and a list of **tagged tokens** such as `K:firm`, `O:black_cherry`, `CUE:palate`.

In [ ]:
def compile_terms(terms):
    """Compile terms into one case-insensitive regex, longest terms first."""
    terms = sorted(set(terms), key=len, reverse=True)
    pattern = r"(?<!\w)(?:" + "|".join(re.escape(t) for t in terms) + r")(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


COLOR_WORDS = r"(?:red|black|white|green|yellow|dark|blue|orange|pink|purple)"
FOOD_WORDS = (
    r"(?:fruits?|cherr(?:y|ies)|berr(?:y|ies)|plums?|currants?|pepper|apples?|pears?|"
    r"peach(?:es)?|herbs?|herbal|olives?|tea|licorice|flowers?|chocolate|figs?|"
    r"grapefruit|raspberr(?:y|ies)|beans?|skins?|skinned|stone|tobacco)"
)

PATTERNS = {
    "COLOR_FOOD": re.compile(r"(?<!\w)" + COLOR_WORDS + r"(?:\s+|-)" + FOOD_WORDS + r"(?!\w)", re.I),
    "COLOR_AS_NAME": re.compile(
        r"(?<!\w)(?:red|white|pink|rosé)(?:\s+|-)(?:wines?|blends?|grapes?|variet(?:y|ies)|ones?)(?!\w)"
        r"|(?<!\w)(?:this|a|the)\s+(?:red|white)(?!\w)",
        re.I,
    ),
    "SHARED": compile_terms(SHARED_AROMA_FLAVOR_TERMS),
    "O_CUE": compile_terms(OLFACTORY_CUES),
    "G_CUE": compile_terms(GUSTATORY_CUES),
    "V": compile_terms(VISUAL_TERMS),
    "A": compile_terms(AUDITORY_TERMS),
    "K": compile_terms(KINESTHETIC_TERMS),
    "O": compile_terms(OLFACTORY_TERMS),
    "G": compile_terms(DIRECT_GUSTATORY_TERMS),
}

CHANNELS = ["V", "A", "K", "O", "G"]


def _token(channel, term):
    return channel + ":" + re.sub(r"[\s-]+", "_", term.lower().strip())


def tag_review(text):
    """text -> (VAKOG counts dict, list of tagged tokens)"""
    text = str(text).lower()
    counts = dict.fromkeys(CHANNELS, 0.0)
    tokens = []

    for clause in re.split(r"[.;!?]", text):
        if not clause.strip():
            continue

        # 1. context cues (style tokens, not counted)
        o_cues = PATTERNS["O_CUE"].findall(clause)
        g_cues = PATTERNS["G_CUE"].findall(clause)
        tokens += [_token("CUE", c) for c in o_cues + g_cues]

        # 2. colour words that name a wine, not a colour
        rest = PATTERNS["COLOR_AS_NAME"].sub(" ", clause)

        # 3. shared descriptors, colour+food phrases first, then remove them
        shared = PATTERNS["COLOR_FOOD"].findall(rest)
        rest = PATTERNS["COLOR_FOOD"].sub(" ", rest)
        shared += PATTERNS["SHARED"].findall(rest)
        rest = PATTERNS["SHARED"].sub(" ", rest)

        if o_cues and not g_cues:
            target = "O"
        elif g_cues and not o_cues:
            target = "G"
        else:
            target = "OG"  # no cue or both cues: genuinely ambiguous

        for term in shared:
            if target == "OG":
                counts["O"] += 0.5
                counts["G"] += 0.5
            else:
                counts[target] += 1
            tokens.append(_token(target, term))

        # 4. single-channel descriptors on what is left
        for ch in CHANNELS:
            for term in PATTERNS[ch].findall(rest):
                counts[ch] += 1
                tokens.append(_token(ch, term))

    return counts, tokens


def vakog_features(text):
    """text -> (dict of dense VAKOG features, space-joined tagged tokens)"""
    counts, tokens = tag_review(text)
    n_words = max(len(re.findall(r"[a-z']+", str(text).lower())), 1)
    total = sum(counts.values())

    row = {f"{c}_rate": counts[c] / n_words for c in CHANNELS}   # per word, not per sensory total
    row["sensory_density"] = total / n_words                     # how sensory the writing is overall

    logs = np.log(np.array([counts[c] for c in CHANNELS]) + 0.5) # centered log-ratio (clr)
    row.update({f"{c}_clr": v for c, v in zip(CHANNELS, logs - logs.mean())})

    return row, " ".join(tokens)


VAKOG_DENSE = [f"{c}_rate" for c in CHANNELS] + ["sensory_density"] + [f"{c}_clr" for c in CHANNELS]

## 6. Sanity-check the parser on an example

In [ ]:
example = """
Aromas include tropical fruit, broom, brimstone and dried herb.
The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
Deep ruby in color, this red shows black cherry and white pepper on the finish.
"""

counts, tokens = tag_review(example)
print("Counts:", counts)
print("Tokens:", tokens)
print()
print(pd.Series(vakog_features(example)[0]).round(3))

## 7. Extract VAKOG features for the full dataset

Reviews with no sensory terms are **kept**: per-word rates are simply 0 (they were undefined as ratios in v1).

In [ ]:
features = df["text_masked"].apply(vakog_features)

vakog_dense_df = pd.DataFrame([f[0] for f in features], index=df.index)
df = pd.concat([df, vakog_dense_df], axis=1)
df["vakog_tokens"] = [f[1] for f in features]

print("Reviews with zero sensory terms:", int((df["sensory_density"] == 0).sum()))
display(df[["text_masked", "taster_name", "vakog_tokens"] + VAKOG_DENSE[:6]].head())

## 8. Investigate the author cutoff

In [ ]:
author_counts = df["taster_name"].value_counts()

cutoff_results = []

for cutoff in [50, 100, 250, 500, 1000, 2000]:
    authors = author_counts[author_counts >= cutoff]

    cutoff_results.append({
        "minimum_reviews": cutoff,
        "known_authors": len(authors),
        "known_reviews": authors.sum(),
        "percent_of_dataset": authors.sum() / len(df) * 100,
    })

cutoff_table = pd.DataFrame(cutoff_results)

display(cutoff_table)

## 9. Select authors

Authors with at least `KNOWN_AUTHOR_MIN_REVIEWS` reviews are modelled. The v1 "unseen author" test is dropped: below the cutoff there are only a handful of people, too few to evaluate rejection reliably (mention it as future work).

In [ ]:
KNOWN_AUTHOR_MIN_REVIEWS = 500

author_counts = df["taster_name"].value_counts()
known_authors = author_counts[author_counts >= KNOWN_AUTHOR_MIN_REVIEWS].index

df_model = df[df["taster_name"].isin(known_authors)].reset_index(drop=True)

print("Authors modelled:", len(known_authors))
print("Reviews modelled:", len(df_model))
print("Excluded authors:", list(author_counts[author_counts < KNOWN_AUTHOR_MIN_REVIEWS].index))

## 10. Train / validation / test split, grouped by winery

`StratifiedGroupKFold` keeps every winery on **one** side of the split while keeping each author's share roughly equal. Result: ~71 / 14 / 14.

In [ ]:
def make_splits(frame, seed=SEED):
    """frame -> (train, val, test). Stratified by author, every winery on one side only."""
    groups = frame["winery"].fillna("unknown winery").to_numpy()
    y = frame["taster_name"].to_numpy()

    outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    trainval_idx, test_idx = next(outer.split(frame, y, groups))

    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    tr, va = next(inner.split(frame.iloc[trainval_idx], y[trainval_idx], groups[trainval_idx]))

    train = frame.iloc[trainval_idx[tr]].copy()
    val = frame.iloc[trainval_idx[va]].copy()
    test = frame.iloc[test_idx].copy()

    assert not (set(train["winery"].dropna()) & set(test["winery"].dropna())), "winery leaked into test"
    assert not (set(train["winery"].dropna()) & set(val["winery"].dropna())), "winery leaked into val"
    return train, val, test


train_df, val_df, test_df = make_splits(df_model)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

display(pd.DataFrame({
    "train %": train_df["taster_name"].value_counts(normalize=True),
    "test %": test_df["taster_name"].value_counts(normalize=True),
}).mul(100).round(1))

## 11. Encode author labels

In [ ]:
label_encoder = LabelEncoder().fit(train_df["taster_name"])

y_train = label_encoder.transform(train_df["taster_name"])
y_val = label_encoder.transform(val_df["taster_name"])
y_test = label_encoder.transform(test_df["taster_name"])

AUTHORS = label_encoder.classes_
N_CLASSES = len(AUTHORS)

for i, author in enumerate(AUTHORS):
    print(i, author)

# FEATURE SET 1 — VAKOG only

* 11 dense features: 5 per-word rates, sensory density, 5 clr log-ratios (standardised)
* + a sparse TF-IDF matrix of VAKOG-tagged tokens (`K:firm`, `O:black_cherry`, `CUE:palate` …)

Everything is fit on **train only**.

In [ ]:
def build_vakog_matrices(train, val, test, min_df=20):
    """(train, val, test) frames -> sparse X_train, X_val, X_test, feature names"""
    tfidf = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=min_df, sublinear_tf=True)
    scaler = StandardScaler()

    def transform(frame, fit=False):
        if fit:
            dense = scaler.fit_transform(frame[VAKOG_DENSE])
            terms = tfidf.fit_transform(frame["vakog_tokens"])
        else:
            dense = scaler.transform(frame[VAKOG_DENSE])
            terms = tfidf.transform(frame["vakog_tokens"])
        return sp.hstack([sp.csr_matrix(dense), terms]).tocsr()

    X_tr = transform(train, fit=True)
    X_va = transform(val)
    X_te = transform(test)
    names = VAKOG_DENSE + list(tfidf.get_feature_names_out())
    return X_tr, X_va, X_te, names


X_train_vakog, X_val_vakog, X_test_vakog, vakog_names = build_vakog_matrices(train_df, val_df, test_df)
print("VAKOG matrix:", X_train_vakog.shape)

# FEATURE SET 2 — Wine characteristics only (control)

Measures how much authorship is predictable just from **what** a critic is assigned to taste. `winery` stays excluded.

In [ ]:
WINE_NUMERIC = ["points", "price"]
WINE_CATEGORICAL = ["country", "province", "region_1", "variety"]


def build_wine_matrices(train, val, test):
    """(train, val, test) frames -> sparse X_train, X_val, X_test, feature names"""
    pre = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), WINE_NUMERIC),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=50)),
        ]), WINE_CATEGORICAL),
    ])
    X_tr = sp.csr_matrix(pre.fit_transform(train))
    X_va = sp.csr_matrix(pre.transform(val))
    X_te = sp.csr_matrix(pre.transform(test))
    return X_tr, X_va, X_te, list(pre.get_feature_names_out())


X_train_wine, X_val_wine, X_test_wine, wine_names = build_wine_matrices(train_df, val_df, test_df)
print("Wine matrix:", X_train_wine.shape)

# FEATURE SET 3 — VAKOG + wine characteristics

In [ ]:
X_train_combined = sp.hstack([X_train_vakog, X_train_wine]).tocsr()
X_val_combined = sp.hstack([X_val_vakog, X_val_wine]).tocsr()
X_test_combined = sp.hstack([X_test_vakog, X_test_wine]).tocsr()
combined_names = vakog_names + wine_names

print("Combined matrix:", X_train_combined.shape)

## 12. Metrics and model-selection rule

* **Optimizing metric:** macro F1 (every author counts equally).
* **Satisficing metric:** recall. Each author's recall must be at least `R_MIN`, so no critic is systematically missed (missed = false negative).
* Among hyper-parameter settings that pass the recall floor, the highest **validation** macro F1 wins. The test set is touched once, at the end.

In [ ]:
R_MIN = 0.25  # recall floor every author must clear; raise it if your models comfortably pass


def evaluate(y_true, y_pred):
    labels = np.unique(y_true)
    per_author_recall = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    return {
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "macro_recall": per_author_recall.mean(),
        "min_author_recall": per_author_recall.min(),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def pick_best(grid_df):
    """Highest macro F1 among settings whose worst author still has recall >= R_MIN."""
    passing = grid_df[grid_df["min_author_recall"] >= R_MIN]
    if passing.empty:
        print(f"  no setting reached recall floor {R_MIN}; using the one with the best worst-author recall")
        return grid_df.sort_values("min_author_recall", ascending=False).iloc[0]
    return passing.sort_values("macro_f1", ascending=False).iloc[0]

## 13. Elastic net multinomial logistic regression (the profile model)

`C` = 1 / regularisation strength, `l1_ratio` = share of L1 (sparsity) vs L2 (shrinkage). `class_weight="balanced"` up-weights small authors.

In [ ]:
ENET_GRID = [{"C": C, "l1_ratio": r} for C in [0.1, 1.0] for r in [0.2, 0.8]]


def make_enet(C, l1_ratio):
    kwargs = dict(
        solver="saga", C=C, l1_ratio=l1_ratio, class_weight="balanced",
        max_iter=1000, tol=1e-3, random_state=SEED,
    )
    # scikit-learn < 1.8 needs penalty="elasticnet"; 1.8+ infers it from l1_ratio
    if tuple(int(p) for p in SKLEARN_VERSION.split(".")[:2]) < (1, 8):
        kwargs["penalty"] = "elasticnet"
    return LogisticRegression(**kwargs)


def fit_enet(X_tr, y_tr, X_va, y_va, grid=ENET_GRID):
    rows, models = [], []
    for params in grid:
        model = make_enet(**params).fit(X_tr, y_tr)
        rows.append({**params, **evaluate(y_va, model.predict(X_va))})
        models.append(model)
    grid_df = pd.DataFrame(rows)
    best = pick_best(grid_df)
    return models[best.name], grid_df

## 14. XGBoost (the accuracy ceiling)

Balanced sample weights handle imbalance. Early stopping on validation log-loss chooses the number of trees.

In [ ]:
XGB_GRID = [{"max_depth": d} for d in [3, 6]]


def fit_xgb(X_tr, y_tr, X_va, y_va, grid=XGB_GRID):
    w_tr = compute_sample_weight("balanced", y_tr)
    w_va = compute_sample_weight("balanced", y_va)
    rows, models = [], []
    for params in grid:
        model = XGBClassifier(
            objective="multi:softprob",
            n_estimators=1000,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            eval_metric="mlogloss",
            early_stopping_rounds=30,
            random_state=SEED,
            n_jobs=-1,
            **params,
        )
        model.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            sample_weight_eval_set=[w_va],
            verbose=False,
        )
        rows.append({**params, "trees_used": model.best_iteration + 1,
                     **evaluate(y_va, model.predict(X_va))})
        models.append(model)
    grid_df = pd.DataFrame(rows)
    best = pick_best(grid_df)
    return models[best.name], grid_df

## 15. Run both models on a feature set

In [ ]:
results = []   # one row per (feature set, model), scored on TEST
fitted = {}    # (feature set, model) -> (fitted model, test predictions)


def run_feature_set(name, X_tr, X_va, X_te):
    print(f"=== {name} ===")
    enet, enet_grid = fit_enet(X_tr, y_train, X_va, y_val)
    print("Elastic net validation grid:")
    display(enet_grid.round(3))

    xgb, xgb_grid = fit_xgb(X_tr, y_train, X_va, y_val)
    print("XGBoost validation grid:")
    display(xgb_grid.round(3))

    for model_name, model in [("Elastic net", enet), ("XGBoost", xgb)]:
        pred = model.predict(X_te)
        results.append({"features": name, "model": model_name, **evaluate(y_test, pred)})
        fitted[(name, model_name)] = (model, pred)

# 16. Train — VAKOG only

In [ ]:
run_feature_set("VAKOG only", X_train_vakog, X_val_vakog, X_test_vakog)

# 17. Train — Wine characteristics only

In [ ]:
run_feature_set("Wine characteristics", X_train_wine, X_val_wine, X_test_wine)

# 18. Train — VAKOG + wine characteristics

In [ ]:
run_feature_set("VAKOG + wine", X_train_combined, X_val_combined, X_test_combined)

## 19. Compare models (test set)

* Left: macro F1 vs. the majority-class baseline.
* Right: worst-author recall vs. the recall floor.

In [ ]:
majority = np.bincount(y_train).argmax()
baseline = {"features": "—", "model": "Majority-class baseline",
            **evaluate(y_test, np.full_like(y_test, majority))}

results_df = pd.DataFrame(results + [baseline])
display(results_df.sort_values("macro_f1", ascending=False).round(3))

FEATURE_ORDER = ["VAKOG only", "Wine characteristics", "VAKOG + wine"]
plot_df = results_df[results_df["model"] != "Majority-class baseline"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, metric, title in zip(
    axes,
    ["macro_f1", "min_author_recall"],
    ["Macro F1 (optimizing)", "Worst-author recall (satisficing)"],
):
    pivot = plot_df.pivot(index="features", columns="model", values=metric).loc[FEATURE_ORDER]
    pivot.plot.bar(ax=ax, rot=0, color=["#4C72B0", "#DD8452"])
    if metric == "macro_f1":
        ax.axhline(baseline[metric], ls=":", c="gray", label="majority baseline")
    else:
        ax.axhline(R_MIN, ls="--", c="red", label=f"recall floor = {R_MIN}")
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.set_xlabel("")
    ax.legend()
plt.tight_layout()
plt.show()

vakog_rows = results_df[results_df["features"] == "VAKOG only"]
VAKOG_BEST = vakog_rows.sort_values("macro_f1").iloc[-1]["model"]
vakog_best_pred = fitted[("VAKOG only", VAKOG_BEST)][1]
print("Best VAKOG-only model:", VAKOG_BEST)

## 20. VAKOG confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    vakog_best_pred,
    display_labels=AUTHORS,
    normalize="true",
    xticks_rotation=90,
    values_format=".2f",
    ax=ax,
    colorbar=False,
)

plt.title(f"VAKOG-only ({VAKOG_BEST}) — row-normalised confusion matrix\n(diagonal = recall per author)")
plt.tight_layout()
plt.show()

## 21. Per-author performance

In [ ]:
print(classification_report(y_test, vakog_best_pred, target_names=AUTHORS, digits=3, zero_division=0))

## 22. Equalized odds audit

For each author *k* and each group *g* (score band, price tier):

* **TPR** = share of *k*'s reviews correctly attributed to *k*
* **FPR** = share of *other* critics' reviews wrongly attributed to *k*

Equalized odds holds when TPR and FPR are about the same in every group. The **gap** is max − min across groups. Groups with fewer than 30 reviews by that author are skipped (too noisy).

Country isn't used as a fairness group: most critics cover one region, so it would measure assignment, not bias.

In [ ]:
MODEL_TO_AUDIT = ("VAKOG only", VAKOG_BEST)
EO_TOLERANCE = 0.10   # gaps above this are flagged


def fairness_groups(frame):
    return pd.DataFrame({
        "score band": pd.cut(frame["points"], [0, 86, 89, 100], labels=["80–86", "87–89", "90–100"]),
        "price tier": pd.cut(frame["price"], [0, 15, 30, 60, np.inf], labels=["<$15", "$15–30", "$30–60", "$60+"]),
    }).reset_index(drop=True)


def equalized_odds_table(y_true, y_pred, groups, class_names, min_count=30):
    rows = []
    for k, author in enumerate(class_names):
        for attribute in groups.columns:
            for level in groups[attribute].cat.categories:
                in_group = (groups[attribute] == level).to_numpy()
                positives = in_group & (y_true == k)
                negatives = in_group & (y_true != k)
                if positives.sum() < min_count or negatives.sum() == 0:
                    continue
                rows.append({
                    "author": author,
                    "attribute": attribute,
                    "group": level,
                    "n_author_reviews": int(positives.sum()),
                    "TPR": (y_pred[positives] == k).mean(),
                    "FPR": (y_pred[negatives] == k).mean(),
                })
    return pd.DataFrame(rows)


audit_pred = fitted[MODEL_TO_AUDIT][1]
test_groups = fairness_groups(test_df)
eo = equalized_odds_table(y_test, audit_pred, test_groups, AUTHORS)

gaps = (
    eo.groupby(["author", "attribute"])
      .agg(TPR_gap=("TPR", lambda s: s.max() - s.min()),
           FPR_gap=("FPR", lambda s: s.max() - s.min()),
           groups_compared=("group", "count"))
      .reset_index()
)
gaps = gaps[gaps["groups_compared"] >= 2].copy()
gaps["EO_gap"] = gaps[["TPR_gap", "FPR_gap"]].max(axis=1)
gaps["passes"] = gaps["EO_gap"] <= EO_TOLERANCE

display(gaps.sort_values("EO_gap", ascending=False).round(3))

In [ ]:
# ---- Figure 1: dumbbell plots. Each dot = one group; gray bar = that author's gap.
attributes = list(test_groups.columns)
fig, axes = plt.subplots(len(attributes), 2, figsize=(14, 0.5 * N_CLASSES * len(attributes) + 2), squeeze=False)

for r, attribute in enumerate(attributes):
    sub = eo[eo["attribute"] == attribute]
    authors = sorted(sub["author"].unique())
    ypos = {a: i for i, a in enumerate(authors)}
    levels = [lvl for lvl in test_groups[attribute].cat.categories if lvl in set(sub["group"])]
    colors = plt.cm.viridis(np.linspace(0, 0.85, len(levels)))

    for c, rate in enumerate(["TPR", "FPR"]):
        ax = axes[r, c]
        for a in authors:
            vals = sub.loc[sub["author"] == a, rate]
            ax.hlines(ypos[a], vals.min(), vals.max(), color="lightgray", lw=6, zorder=1)
        for level, color in zip(levels, colors):
            pts = sub[sub["group"] == level]
            ax.scatter(pts[rate], pts["author"].map(ypos), color=color, s=55, label=level, zorder=2)
        ax.set_yticks(range(len(authors)))
        ax.set_yticklabels(authors)
        ax.set_title(f"{rate} by {attribute}")
        ax.set_xlabel(rate + (" (higher = author recognised)" if rate == "TPR" else " (lower = fewer false accusations)"))
        ax.legend(title=attribute, fontsize=8, loc="best")
        ax.grid(axis="x", alpha=0.3)

fig.suptitle(f"Equalized odds — {MODEL_TO_AUDIT[0]} ({MODEL_TO_AUDIT[1]}). Shorter gray bars = fairer", y=1.0)
plt.tight_layout()
plt.show()

# ---- Figure 2: one-glance heatmap of the equalized-odds gap
pivot = gaps.pivot(index="author", columns="attribute", values="EO_gap")
fig, ax = plt.subplots(figsize=(6, 0.45 * len(pivot) + 1.5))
im = ax.imshow(pivot.values, cmap="Reds", vmin=0, vmax=max(0.3, np.nanmax(pivot.values)), aspect="auto")
ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            mark = "✓" if v <= EO_TOLERANCE else "✗"
            ax.text(j, i, f"{v:.2f} {mark}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="EO gap = max(TPR gap, FPR gap)")
ax.set_title(f"Equalized-odds gap (✓ ≤ {EO_TOLERANCE})")
plt.tight_layout()
plt.show()

## 23. Which critics are easy vs. hard to detect?

Per-author F1 on the test set, VAKOG features only.

In [ ]:
per_author_f1 = pd.Series(
    f1_score(y_test, vakog_best_pred, labels=np.arange(N_CLASSES), average=None, zero_division=0),
    index=AUTHORS,
).sort_values()

fig, ax = plt.subplots(figsize=(8, 0.4 * N_CLASSES + 1.5))
colors = ["#C44E52" if v < per_author_f1.median() else "#55A868" for v in per_author_f1]
per_author_f1.plot.barh(ax=ax, color=colors)
ax.axvline(per_author_f1.mean(), ls="--", c="black", label=f"macro F1 = {per_author_f1.mean():.2f}")
ax.set_xlabel("F1 (VAKOG only)")
ax.set_title("Distinct (green) vs. generic (red) writing style")
ax.legend()
plt.tight_layout()
plt.show()

## 24. Reviewer profiles

**(a) VAKOG fingerprint:** average per-word rate of each channel, z-scored across critics (red = uses that sense more than peers).

**(b) Signature words:** the elastic-net terms with the largest positive coefficient for each author, per channel.

In [ ]:
enet_vakog = fitted[("VAKOG only", "Elastic net")][0]
coef = pd.DataFrame(enet_vakog.coef_, index=AUTHORS, columns=vakog_names)

# (a) fingerprint heatmap
rate_cols = [f"{c}_rate" for c in CHANNELS] + ["sensory_density"]
profile = train_df.groupby("taster_name")[rate_cols].mean()
z = (profile - profile.mean()) / profile.std()

fig, ax = plt.subplots(figsize=(8, 0.45 * len(z) + 1.5))
im = ax.imshow(z.values, cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="auto")
ax.set_xticks(range(len(rate_cols)))
ax.set_xticklabels(["Visual", "Auditory", "Kinesthetic", "Olfactory", "Gustatory", "Sensory\ndensity"])
ax.set_yticks(range(len(z)))
ax.set_yticklabels(z.index)
for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        ax.text(j, i, f"{z.values[i, j]:+.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="z-score vs. other critics")
ax.set_title("VAKOG fingerprint per critic")
plt.tight_layout()
plt.show()

# (b) signature words
rows = []
for author in AUTHORS:
    row = {"author": author}
    for ch in ["V", "K", "O", "G", "OG", "CUE"]:
        cols = [c for c in coef.columns if c.startswith(ch + ":")]
        top = coef.loc[author, cols].sort_values(ascending=False).head(3)
        top = top[top > 0]
        row[ch] = ", ".join(t.split(":", 1)[1].replace("_", " ") for t in top.index) or "—"
    rows.append(row)

signature_words = pd.DataFrame(rows).set_index("author")
display(signature_words)

## 25. Confound check: California only

Several critics review California wines, so region is held roughly constant. If VAKOG still beats the wine-only control here, the signal is **style**, not assignment.

In [ ]:
CA_MIN_REVIEWS = 300

ca = df_model[df_model["province"] == "California"]
ca_counts = ca["taster_name"].value_counts()
display(ca_counts)

ca_authors = ca_counts[ca_counts >= CA_MIN_REVIEWS].index
ca = ca[ca["taster_name"].isin(ca_authors)].reset_index(drop=True)

if len(ca_authors) < 2:
    print("Fewer than 2 critics pass the California cutoff; lower CA_MIN_REVIEWS.")
else:
    ca_tr, ca_va, ca_te = make_splits(ca)
    ca_le = LabelEncoder().fit(ca_tr["taster_name"])
    ca_y = [ca_le.transform(f["taster_name"]) for f in (ca_tr, ca_va, ca_te)]

    Xv = build_vakog_matrices(ca_tr, ca_va, ca_te)[:3]
    Xw = build_wine_matrices(ca_tr, ca_va, ca_te)[:3]
    Xc = [sp.hstack([a, b]).tocsr() for a, b in zip(Xv, Xw)]

    ca_rows = []
    for name, X in [("VAKOG only", Xv), ("Wine characteristics", Xw), ("VAKOG + wine", Xc)]:
        model, _ = fit_enet(X[0], ca_y[0], X[1], ca_y[1])
        ca_rows.append({"features": name, **evaluate(ca_y[2], model.predict(X[2]))})
    ca_majority = np.bincount(ca_y[0]).argmax()
    ca_rows.append({"features": "Majority baseline",
                    **evaluate(ca_y[2], np.full_like(ca_y[2], ca_majority))})

    ca_results = pd.DataFrame(ca_rows).set_index("features")
    display(ca_results.round(3))

    ca_results["macro_f1"].plot.barh(color=["#4C72B0", "#8C8C8C", "#55A868", "#CCCCCC"], figsize=(7, 3))
    plt.xlabel("Macro F1 (California critics, elastic net)")
    plt.title("Style vs. assignment, region held constant")
    plt.tight_layout()
    plt.show()

## 26. Bias–variance: learning curve

Best elastic-net settings, trained on growing stratified fractions of the training set.

* Train and validation both low and close → **underfitting** (add features / loosen regularisation).
* Train high, validation low → **overfitting** (add data / tighten regularisation).

In [ ]:
rng = np.random.default_rng(SEED)
lc_rows = []

for fraction in [0.05, 0.1, 0.25, 0.5, 1.0]:
    idx = np.concatenate([
        rng.choice(np.where(y_train == k)[0],
                   size=max(int(fraction * (y_train == k).sum()), 5),
                   replace=False)
        for k in range(N_CLASSES)
    ])
    model = make_enet(enet_vakog.C, enet_vakog.l1_ratio).fit(X_train_vakog[idx], y_train[idx])
    lc_rows.append({
        "n_train": len(idx),
        "train_macro_f1": f1_score(y_train[idx], model.predict(X_train_vakog[idx]), average="macro"),
        "val_macro_f1": f1_score(y_val, model.predict(X_val_vakog), average="macro"),
    })

lc = pd.DataFrame(lc_rows)
display(lc.round(3))

plt.figure(figsize=(7, 4))
plt.plot(lc["n_train"], lc["train_macro_f1"], "o-", label="train")
plt.plot(lc["n_train"], lc["val_macro_f1"], "o-", label="validation")
plt.fill_between(lc["n_train"], lc["val_macro_f1"], lc["train_macro_f1"], alpha=0.15, label="gap = variance")
plt.xscale("log")
plt.xlabel("training reviews (log scale)")
plt.ylabel("macro F1")
plt.title("Learning curve — VAKOG elastic net")
plt.legend()
plt.tight_layout()
plt.show()

# Interpretation guide

1. **Does VAKOG beat the majority baseline on macro F1?** If yes, sensory language carries author information.
2. **Does every author clear the recall floor?** If not, name the critics the model can't find; they have the most generic style.
3. **VAKOG + wine vs. wine only.** A gain means VAKOG adds information beyond assignment patterns.
4. **California check.** VAKOG beating wine-only *with region held constant* is the strongest evidence of personal style.
5. **Equalized odds.** Small gaps mean the model recognises each critic equally well on cheap and expensive, low- and high-scored wines. Large gaps say a critic's language shifts with price or score, which is itself a finding for Winetitude.

**Framing for Winetitude:** high authorship accuracy shows a review reflects the *critic* as much as the *wine*. That supports the "one person's palate" critique. It does **not** by itself prove bias or corruption; pair it with the equalized-odds results and (with Team Douro) score differences between critics on comparable wines.

> Using only VAKOG sensory language, the {model} reached **macro F1 = X** (baseline **B**) with every critic's recall ≥ **R**. Within California, VAKOG reached **Y** vs. **W** for wine characteristics alone, indicating that sensory language **does / does not** carry a personal signature beyond assignment.